Writing code that works is the first goal. Writing code that works *efficiently* is the second — and often more important — one. Complexity analysis gives you a precise, language-agnostic way to measure how an algorithm's resource usage grows as its input grows. It is the shared vocabulary that lets engineers compare approaches, predict behaviour at scale, and make informed trade-offs.

## Why Measure Performance?

Suppose you have a list of one million user records and you need to find one by name. You could scan from the first record to the last — that works. But on average you will check 500,000 records before finding the right one. Double the dataset and you double the work. At a billion records, this approach becomes unusable.

Alternatively, if the list is sorted, you could use a strategy that eliminates half the remaining records on every step. At a billion records, that strategy takes about 30 steps — not 500 million.

Both approaches produce the correct answer. The difference is *how they scale*. Complexity analysis is the tool that makes this difference visible and quantifiable — before you write a single line of production code.

## Big O Notation

**Big O notation** describes how an algorithm's runtime or memory usage grows *relative to the size of its input*. The input size is conventionally called `n`.

Three things to internalise about Big O:

1. **It describes growth, not absolute speed.** An O(n) algorithm on fast hardware might beat an O(log n) algorithm on slow hardware for small inputs. Big O tells you what happens as `n` gets large.
2. **It drops constants and lower-order terms.** `3n + 100` and `n` are both O(n). At large scale, the `3` and `100` become irrelevant compared to the `n`.
3. **It usually describes the worst case.** Unless stated otherwise, Big O refers to the upper bound — the worst scenario your algorithm might encounter.

![Big O Growth Chart](https://raw.githubusercontent.com/schemabotview/dsa/main/img/big-o-growth-chart.svg)

## The Common Complexity Classes

### O(1) — Constant Time

The operation takes the same amount of time regardless of input size. Accessing an element in an array by index is O(1) — the address is computed directly from the index, no searching needed.

### Python

In [ ]:
nums = [10, 20, 30, 40, 50]

# O(1) — direct address calculation, no loop
first = nums[0]
last  = nums[-1]
mid   = nums[len(nums) // 2]

print(first, last, mid)  # 10 50 30

### Kotlin

```kotlin
val nums = listOf(10, 20, 30, 40, 50)

// O(1) — index lookup computes address directly
val first = nums[0]
val last  = nums[nums.size - 1]
val mid   = nums[nums.size / 2]

println("$first $last $mid")  // 10 50 30
```

### O(log n) — Logarithmic Time

Each step eliminates a fixed *fraction* of the remaining work — typically half. The classic example is binary search: at every step you halve the search space. A dataset of one billion elements needs at most 30 steps. Logarithmic algorithms are extremely efficient at scale.

> A quick intuition: if doubling `n` only adds *one more step*, the algorithm is O(log n).

### Python

In [ ]:
# Binary search — O(log n)
# At each step we discard half the remaining elements.
def binary_search(arr: list[int], target: int) -> int:
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1   # discard left half
        else:
            right = mid - 1  # discard right half
    return -1

sorted_nums = list(range(0, 1_000_000, 2))  # 500k even numbers
print(binary_search(sorted_nums, 999_998))  # finds it in ~19 steps

### Kotlin

```kotlin
// Binary search — O(log n)
fun binarySearch(arr: List<Int>, target: Int): Int {
    var left = 0
    var right = arr.size - 1
    while (left <= right) {
        val mid = (left + right) / 2
        when {
            arr[mid] == target -> return mid
            arr[mid] < target  -> left  = mid + 1
            else               -> right = mid - 1
        }
    }
    return -1
}
```

### O(n) — Linear Time

The work grows in direct proportion to the input size. Scanning every element once is O(n). Double the input, double the work. Linear is very often acceptable — sometimes unavoidable.

### Python

In [ ]:
# Linear search — O(n)
# In the worst case we check every element.
def linear_search(arr: list[int], target: int) -> int:
    for i, val in enumerate(arr):
        if val == target:
            return i
    return -1

# Finding the max also requires one full pass — O(n)
def find_max(arr: list[int]) -> int:
    maximum = arr[0]
    for val in arr:
        if val > maximum:
            maximum = val
    return maximum

data = [3, 1, 4, 1, 5, 9, 2, 6]
print(linear_search(data, 9))  # 5
print(find_max(data))          # 9

### Kotlin

```kotlin
// Linear search — O(n)
fun linearSearch(arr: List<Int>, target: Int): Int {
    for ((i, value) in arr.withIndex()) {
        if (value == target) return i
    }
    return -1
}

// Finding the max — also O(n)
fun findMax(arr: List<Int>): Int = arr.reduce { max, v -> if (v > max) v else max }
```

### O(n log n) — Linearithmic Time

This is the complexity of the most efficient general-purpose sorting algorithms — merge sort, heap sort, and Timsort (Python's built-in sort). It is slightly worse than linear but far better than quadratic. At one million elements, O(n log n) means about 20 million operations; O(n²) means one trillion.

### Python

In [ ]:
# Merge sort — O(n log n)
# Divide in half (log n levels), merge each level (n work per level).
def merge_sort(arr: list[int]) -> list[int]:
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left: list[int], right: list[int]) -> list[int]:
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]);  i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

print(merge_sort([5, 3, 8, 1, 9, 2]))  # [1, 2, 3, 5, 8, 9]

### Kotlin

```kotlin
// Merge sort — O(n log n)
fun mergeSort(arr: List<Int>): List<Int> {
    if (arr.size <= 1) return arr
    val mid   = arr.size / 2
    val left  = mergeSort(arr.subList(0, mid))
    val right = mergeSort(arr.subList(mid, arr.size))
    return merge(left, right)
}

fun merge(left: List<Int>, right: List<Int>): List<Int> {
    val result = mutableListOf<Int>()
    var i = 0; var j = 0
    while (i < left.size && j < right.size) {
        if (left[i] <= right[j]) result.add(left[i++])
        else                     result.add(right[j++])
    }
    return result + left.drop(i) + right.drop(j)
}
```

### O(n²) — Quadratic Time

Usually the result of a nested loop where each loop runs `n` times. For every element you process all other elements. Manageable at small sizes (hundreds), painful at thousands, unusable at millions.

### Python

In [ ]:
# Bubble sort — O(n²)
# The outer loop runs n times; the inner loop also runs ~n times.
def bubble_sort(arr: list[int]) -> list[int]:
    arr = arr.copy()
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

print(bubble_sort([5, 3, 8, 1, 9, 2]))  # [1, 2, 3, 5, 8, 9]

### Kotlin

```kotlin
// Bubble sort — O(n²)
fun bubbleSort(arr: MutableList<Int>): List<Int> {
    val a = arr.toMutableList()
    for (i in a.indices) {
        for (j in 0 until a.size - i - 1) {
            if (a[j] > a[j + 1]) {
                val tmp = a[j]; a[j] = a[j + 1]; a[j + 1] = tmp
            }
        }
    }
    return a
}
```

### O(2ⁿ) — Exponential Time

Adding one element to the input *doubles* the work. Naive recursive solutions to problems like generating all subsets or brute-force password cracking fall here. Exponential algorithms are only practical for very small inputs (roughly `n < 25`).

### Python

In [ ]:
# All subsets — O(2^n)
# Each element is either included or excluded → 2^n combinations.
def all_subsets(nums: list) -> list:
    if not nums:
        return [[]]
    rest = all_subsets(nums[1:])
    return rest + [[nums[0]] + s for s in rest]

print(all_subsets([1, 2, 3]))
# [[], [3], [2], [2, 3], [1], [1, 3], [1, 2], [1, 2, 3]]  — 8 subsets for n=3

### Kotlin

```kotlin
// All subsets — O(2^n)
fun allSubsets(nums: List<Int>): List<List<Int>> {
    if (nums.isEmpty()) return listOf(emptyList())
    val rest = allSubsets(nums.drop(1))
    return rest + rest.map { listOf(nums[0]) + it }
}
```

## Time vs Space Complexity

Big O applies to two independent resources:

- **Time complexity** — how many operations does the algorithm perform as `n` grows?
- **Space complexity** — how much *extra* memory does the algorithm use as `n` grows?

The memory used by the input itself is not counted — only the *additional* memory the algorithm allocates. This is called **auxiliary space**.

The two are often in tension. A common trade-off: use extra memory (a hash map) to avoid redundant computation and reduce time from O(n²) to O(n). Whether that trade-off is worth it depends on the constraints of your system.

### Python

In [ ]:
# Two-sum: find indices of two numbers that add to target

# Approach 1 — O(n²) time, O(1) space
def two_sum_brute(nums: list[int], target: int) -> tuple[int, int]:
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return (i, j)

# Approach 2 — O(n) time, O(n) space  (trade memory for speed)
def two_sum_hash(nums: list[int], target: int) -> tuple[int, int]:
    seen = {}                        # extra O(n) space
    for i, val in enumerate(nums):
        complement = target - val
        if complement in seen:
            return (seen[complement], i)
        seen[val] = i

data = [2, 7, 11, 15]
print(two_sum_brute(data, 9))  # (0, 1)
print(two_sum_hash(data, 9))   # (0, 1)

### Kotlin

```kotlin
// Approach 1 — O(n²) time, O(1) space
fun twoSumBrute(nums: List<Int>, target: Int): Pair<Int, Int>? {
    for (i in nums.indices)
        for (j in i + 1 until nums.size)
            if (nums[i] + nums[j] == target) return Pair(i, j)
    return null
}

// Approach 2 — O(n) time, O(n) space
fun twoSumHash(nums: List<Int>, target: Int): Pair<Int, Int>? {
    val seen = mutableMapOf<Int, Int>()     // extra O(n) space
    for ((i, value) in nums.withIndex()) {
        val complement = target - value
        if (complement in seen) return Pair(seen[complement]!!, i)
        seen[value] = i
    }
    return null
}
```

## Best, Worst, and Average Case

The same algorithm can behave very differently depending on the input it receives:

| Case | Meaning | Notation |
|---|---|---|
| **Best case** | The most favourable input possible | Ω (Omega) |
| **Worst case** | The most demanding input possible | O (Big O) |
| **Average case** | Expected performance across typical inputs | Θ (Theta) |

In practice, **worst case is what matters most** — it is the guarantee you can give. When someone asks what the complexity of your algorithm is, they almost always mean worst case.

Linear search is a good illustration: best case is O(1) (the target is the first element), worst case is O(n) (the target is last or not present at all), and average case is O(n/2) — which simplifies to O(n).

## How to Analyse Code

Four practical rules cover the vast majority of cases:

**1. A single loop over `n` elements → O(n)**

**2. Nested loops over `n` elements → O(n²)** — and three loops deep gives O(n³), and so on.

**3. Halving the problem at each step → O(log n)**

**4. Sequential (non-nested) blocks add; keep the dominant term**
— O(n) + O(n²) = O(n²). The lower term is dropped.

### Python

In [ ]:
def analyse_me(arr: list[int]) -> None:
    # Block A — O(n): one pass to find the max
    maximum = arr[0]
    for val in arr:
        if val > maximum:
            maximum = val

    # Block B — O(n²): nested loop to find all pairs
    pairs = []
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            pairs.append((arr[i], arr[j]))

    # Total: O(n) + O(n²) = O(n²)  — dominant term wins
    print(f"max={maximum}, pairs={len(pairs)}")

analyse_me([3, 1, 4, 1, 5])  # max=5, pairs=10

### Kotlin

```kotlin
fun analyseMe(arr: List<Int>) {
    // Block A — O(n)
    val maximum = arr.max()

    // Block B — O(n²)
    val pairs = mutableListOf<Pair<Int,Int>>()
    for (i in arr.indices)
        for (j in i + 1 until arr.size)
            pairs.add(Pair(arr[i], arr[j]))

    // Total: O(n) + O(n²) = O(n²)
    println("max=$maximum, pairs=${pairs.size}")
}
```

## Complexity Cheat Sheet

| Complexity | Name | Example | Feasible at n = 10⁶? |
|---|---|---|---|
| O(1) | Constant | Array index access | ✅ |
| O(log n) | Logarithmic | Binary search | ✅ |
| O(n) | Linear | Linear scan, find max | ✅ |
| O(n log n) | Linearithmic | Merge sort, Timsort | ✅ |
| O(n²) | Quadratic | Bubble sort, nested loops | ⚠️ slow |
| O(2ⁿ) | Exponential | All subsets, brute-force | ❌ |
| O(n!) | Factorial | All permutations | ❌ |

## Key Takeaways

- **Big O** describes how an algorithm scales with input size — it measures growth, not raw speed.
- Constants and lower-order terms are dropped: `5n + 200` is O(n).
- The hierarchy from best to worst: O(1) → O(log n) → O(n) → O(n log n) → O(n²) → O(2ⁿ).
- **Time and space complexity are independent** — a faster algorithm often costs more memory.
- **Worst case is the default** unless you have a strong reason to reason about average case.
- Nested loops usually signal O(n²). Halving the problem usually signals O(log n). Sequential blocks keep the dominant term.

From here on, every data structure and algorithm in this course comes with a complexity analysis. The vocabulary you built here is the lens you will use to evaluate every trade-off.